# 备件输机（应联动）未联动清单

从  中筛选指定时间范围的审批通过记录，筛选已关联量产件、有对应零件裸价但未联动的备件记录。


In [ ]:
import pandas as pd
from datetime import datetime
import os

# ===== 参数配置 - 请修改这里的起止日期 =====
START_DATE = "2026-05-30"   # 起始日期 (格式: YYYY-MM-DD)
END_DATE = "2026-06-12"     # 结束日期 (格式: YYYY-MM-DD)
# ==========================================

INPUT_FILE = "../data/eps3_spare.xlsx"
OUTPUT_DIR = "../output/备件输机(应联动)未联动清单"
DATE_SUFFIX = datetime.now().strftime("%Y%m%d")
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"备件输机(应联动)_未联动_{DATE_SUFFIX}.xlsx")


In [ ]:
# 1. 读取数据
df = pd.read_excel(INPUT_FILE)
print(f"读取总行数: {len(df)}")

# 2. 按审批通过时间筛选
df["审批通过时间"] = pd.to_datetime(df["审批通过时间"].str[0:10], errors="coerce")
mask = (df["审批通过时间"] >= START_DATE) & (df["审批通过时间"] <= END_DATE)
df = df[mask].copy()
print(f"审批通过时间筛选 ({START_DATE} ~ {END_DATE}): {len(df)} 行")


In [ ]:
# 3. 筛选"量产件号"不为空
df = df.dropna(subset=["量产件号"]).copy()
print(f"量产件号不为空: {len(df)} 行")

# 4. 筛选"对应零件裸价"不为空
df = df.dropna(subset=["对应零件裸价"]).copy()
print(f"对应零件裸价不为空: {len(df)} 行")

# 5. 筛选"是否联动(0是，1否)" == 1（未联动）
df = df[df["是否联动(0是，1否)"] == 1].copy()
print(f"未联动记录: {len(df)} 行")


In [ ]:
# 6. 保留所需字段并重命名
df_output = df[[
    "价格通知书号", "供应商编码", "供应商名称", "备件号", "备件名称",
    "备件裸价", "生效日期", "失效日期", "量产件号", "对应零件裸价",
    "采购工程师姓名"
]].copy()

# 重命名生效/失效日期
df_output.rename(columns={
    "生效日期": "备件生效",
    "失效日期": "备件失效"
}, inplace=True)

print(f"输出记录数: {len(df_output)}")

# 7. 导出到 Excel
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_output.to_excel(OUTPUT_FILE, index=False)
print(f"结果已导出: {OUTPUT_FILE}")
